In [ ]:
# !pip install google-cloud-texttospeech

In [ ]:
import io
import ollama
from PIL import Image
from google.cloud import texttospeech

### Qwen 2.5 for Image Caption

In [ ]:
### Ref https://ollama.com/library/qwen2.5vl:7b
# to download the model, we need to open terminal and run: ollama pull qwen2.5vl:7b
# or ollama run qwen2.5vl:7b
# check whether it's downloaded: ollama list

In [ ]:
image_path = 'example_photo/img_1.jpg'
image = Image.open(image_path)

caption_prompt = """
You are a compassionate storyteller. Using the attached photo, craft a 120–180 word micro-story
intended for an older adult and their family. The story should gently evoke memories, spark warm
conversation, and support emotional well-being.

Guidelines:
- Focus on mood, place, season, and relationships.
- Use 2–3 sensory details (sounds, scents, textures, light).
- Use warm, respectful language and short, vivid sentences.
- Avoid names, ages, or locations. Use tentative phrasing ("perhaps", "it seems").
- Emphasize connection and small rituals.
"""

caption_response = ollama.chat(
    model="qwen2.5-vl",
    messages=[
        {
            "role": "user",
            "content": caption_prompt,
            "images": [image_path],
        }
    ]
)

caption_text = caption_resp["message"]["content"]
print("Caption:", caption_text)

### CantoneseLLMChat for Translation

In [ ]:
### Ref https://ollama.com/humblemat/hon9kon9ize_CantoneseLLMChat-v1.0-7B-F16.gguf.q6_k
# to download the model, we need to open terminal and run: ollama run humblemat/hon9kon9ize_CantoneseLLMChat-v1.0-7B-F16.gguf.q6_k
# check whether it's downloaded: ollama list

In [ ]:
image_story = """In the soft light of a late afternoon, a family stands together on a bridge, their laughter mingling with the gentle breeze. The city skyline stretches behind them, a tapestry of towering skyscrapers and twinkling lights. The woman, her arm around the child, points towards something in the distance, perhaps a familiar landmark or a new adventure. The man beside her smiles warmly, his hand gently resting on the child's shoulder. The scene is one of simple joy and shared moments, a snapshot of life's little treasures. As they look out over the water, the family seems to be building memories that will last a lifetime, each moment a testament to the love and connection that binds them."""

cantonese_prompt = (f"Translate the below text into Cantonese: {image_story}")

cantonese_resp = ollama.chat(
    model="humblemat/hon9kon9ize_CantoneseLLMChat-v1.0-7B-F16.gguf.q6_k",
    messages=[{"role": "user", "content": cantonese_prompt}]
)

cantonese_text = cantonese_resp["message"]["content"]
print("Cantonese:", cantonese_text)

### Google for TTS

In [ ]:
# I can't find google TTS on ollama so we stick to Google TTS API

In [ ]:
cantonese_text = "喺昏黄嘅夕陽下，一家大細企喺橋上面，佢哋嘅笑聲同微風交織埋一齊。城市嘅天際線喺佢哋後面延伸，形成一幅高樓大廈同閃閃發光嘅燈光交織而成嘅畫布。個女人嘅手臂環繞住個細路，指住遠處嘅某樣嘢，可能係一個熟悉嘅地標或者一個新嘅冒險。個男人企喺佢隔離，笑得好開心，隻手輕輕噉搭喺個細路嘅膊頭上。呢個畫面係快樂同共享時光嘅簡單喜悅，一張生活中小寶藏嘅快照。當佢哋望住下面嘅水，呢個家庭似乎喺度創造緊可以令成世都記得嘅回憶，每個時刻都係對嗰種將佢哋緊緊縛埋一齊嘅愛同聯繫嘅證明。"

client = texttospeech.TextToSpeechClient.from_service_account_json("YOUR_API.json")

synthesis_input = texttospeech.SynthesisInput(text=cantonese_text)

voice = texttospeech.VoiceSelectionParams(
    language_code="yue-HK",
    name="yue-HK-Standard-A"
)

audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)

response = client.synthesize_speech(
    input=synthesis_input, voice=voice, audio_config=audio_config
)

with open("cantonese_output.mp3", "wb") as out:
    out.write(response.audio_content)
